In [13]:
import os
import pandas as pd

ruta_raw = "../data/raw"

archivos = [
    "Datos de la vivienda.csv",
    "Servicios del hogar.csv",
    "Tecnologías de información y comunicación.csv"
]

print("VERIFICACIÓN DE ARCHIVOS EN data/raw")
for nombre in archivos:
    ruta_completa = os.path.join(ruta_raw, nombre)
    existe = os.path.exists(ruta_completa)
    tamano_mb = os.path.getsize(ruta_completa) / (1024 * 1024) if existe else 0
    print(f"Archivo: {nombre} | Existe: {existe} | Tamaño: {tamano_mb:.2f} MB")

VERIFICACIÓN DE ARCHIVOS EN data/raw
Archivo: Datos de la vivienda.csv | Existe: True | Tamaño: 8.78 MB
Archivo: Servicios del hogar.csv | Existe: True | Tamaño: 18.36 MB
Archivo: Tecnologías de información y comunicación.csv | Existe: True | Tamaño: 29.86 MB


In [14]:
df_vivienda = pd.read_csv(os.path.join(ruta_raw, "Datos de la vivienda.csv"), sep=";", decimal=",", low_memory=False)
df_hogar = pd.read_csv(os.path.join(ruta_raw, "Servicios del hogar.csv"), sep=";", decimal=",", low_memory=False)
df_tic = pd.read_csv(os.path.join(ruta_raw, "Tecnologías de información y comunicación.csv"), sep=";", decimal=",", low_memory=False)

print("DIMENSIONES DE CADA MÓDULO")
print(f"1. Vivienda:  {df_vivienda.shape[0]:,} filas x {df_vivienda.shape[1]} columnas")
print(f"2. Hogar:     {df_hogar.shape[0]:,} filas x {df_hogar.shape[1]} columnas")
print(f"3. TIC:       {df_tic.shape[0]:,} filas x {df_tic.shape[1]} columnas")

print("\nVERIFICACIÓN DE LLAVES PRIMARIAS")
print("Vivienda - DIRECTORIO únicos:", df_vivienda["DIRECTORIO"].nunique())
print("Hogar    - pares únicos (DIRECTORIO, ORDEN):", len(df_hogar.drop_duplicates(["DIRECTORIO", "ORDEN"])))
print("TIC      - registros únicos (DIRECTORIO, SECUENCIA_P, ORDEN):", len(df_tic.drop_duplicates(["DIRECTORIO", "SECUENCIA_P", "ORDEN"])))

print("\nTIPO DE DATO DE FEX_C")
print("FEX_C en TIC:", df_tic["FEX_C"].dtype, "| Ejemplo:", df_tic["FEX_C"].iloc[0])

DIMENSIONES DE CADA MÓDULO
1. Vivienda:  86,848 filas x 46 columnas
2. Hogar:     87,060 filas x 98 columnas
3. TIC:       221,043 filas x 89 columnas

VERIFICACIÓN DE LLAVES PRIMARIAS
Vivienda - DIRECTORIO únicos: 86848
Hogar    - pares únicos (DIRECTORIO, ORDEN): 87060
TIC      - registros únicos (DIRECTORIO, SECUENCIA_P, ORDEN): 221043

TIPO DE DATO DE FEX_C
FEX_C en TIC: float64 | Ejemplo: 472.1319014


In [15]:
# 1. Seleccionamos las columnas clave 
cols_vivienda = ["DIRECTORIO", "P1_DEPARTAMENTO", "P1_MUNICIPIO", "REGION", "CLASE"]

cols_hogar = ["DIRECTORIO", "ORDEN", "I_HOGAR", "PERCAPITA", "CANT_PERSONAS_HOGAR"]

cols_tic = [
    "DIRECTORIO", "SECUENCIA_ENCUESTA", "SECUENCIA_P", "ORDEN", "FEX_C",
    "P1910",    # Uso de computador de escritorio
    "P1911",    # Uso de computador portátil
    "P1912",    # Uso de tableta
    "P1084",    # Frecuencia de uso de internet
    "P765S1",   # Acceso internet: PC escritorio
    "P765S2",   # Acceso internet: Portátil
    "P765S3",   # Acceso internet: Tableta
    "P765S4",   # Acceso internet: Celular
    "P1085S1",  # Lugar acceso: Hogar
    "P1085S2",  # Lugar acceso: Trabajo
    "P1085S3",  # Lugar acceso: Institución educativa
    "P1929",    # Razón principal por la que NO usa internet
    "P1082",    # Uso de teléfono celular
    "P804"      # Frecuencia uso redes sociales
]

# 2. Unimos TIC (personas) + Vivienda por DIRECTORIO (Relación N:1)
df_unido = pd.merge(
    df_tic[cols_tic],
    df_vivienda[cols_vivienda],
    on="DIRECTORIO",
    how="left"
)

# 3. Unimos Resultado + Servicios del hogar por (DIRECTORIO, SECUENCIA_P) == (DIRECTORIO, ORDEN) (Relación N:1)
df_unido = pd.merge(
    df_unido,
    df_hogar[cols_hogar],
    left_on=["DIRECTORIO", "SECUENCIA_P"],
    right_on=["DIRECTORIO", "ORDEN"],
    how="left",
    suffixes=("", "_HOGAR")
)

# Eliminamos la columna auxiliar ORDEN_HOGAR que nos sirvió solo para el cruce
df_unido = df_unido.drop(columns=["ORDEN_HOGAR"])

print("AUDITORÍA DE LA INTEGRACIÓN DE LOS 3 MÓDULOS")
print(f"Filas originales en TIC:  {len(df_tic):,}")
print(f"Filas después del cruce:  {len(df_unido):,}")
print(f"Total de columnas unidas: {df_unido.shape[1]}")
print(f"Nulos en Departamento:    {df_unido['P1_DEPARTAMENTO'].isna().sum()}")
print(f"Nulos en Ingreso PerCap:  {df_unido['PERCAPITA'].isna().sum()}")

AUDITORÍA DE LA INTEGRACIÓN DE LOS 3 MÓDULOS
Filas originales en TIC:  221,043
Filas después del cruce:  221,043
Total de columnas unidas: 26
Nulos en Departamento:    0
Nulos en Ingreso PerCap:  0


In [12]:
mapa_departamentos = {
    5: "Antioquia", 8: "Atlántico", 11: "Bogotá D.C.", 13: "Bolívar",
    15: "Boyacá", 17: "Caldas", 18: "Caquetá", 19: "Cauca",
    20: "Cesar", 23: "Córdoba", 25: "Cundinamarca", 27: "Chocó",
    41: "Huila", 44: "La Guajira", 47: "Magdalena", 50: "Meta",
    52: "Nariño", 54: "Norte de Santander", 63: "Quindío", 66: "Risaralda",
    68: "Santander", 70: "Sucre", 73: "Tolima", 76: "Valle del Cauca",
    81: "Arauca", 85: "Casanare", 86: "Putumayo", 88: "San Andrés y Providencia",
    91: "Amazonas", 94: "Guainía", 95: "Guaviare", 97: "Vaupés", 99: "Vichada"
}

df_unido["NOMBRE_DEPARTAMENTO"] = df_unido["P1_DEPARTAMENTO"].map(mapa_departamentos)
df_unido["ZONA"] = df_unido["CLASE"].map({1: "Urbano", 2: "Rural"})

ruta_salida = "../data/processed/dataset_brecha_unificado.csv"
df_unido.to_csv(ruta_salida, index=False, encoding="utf-8")

tamano_mb = os.path.getsize(ruta_salida) / (1024 * 1024)
print(f"Archivo guardado exitosamente en: {ruta_salida}")
print(f"Dimensiones finales: {df_unido.shape[0]:,} filas x {df_unido.shape[1]} columnas")
print(f"Tamaño del archivo unificado: {tamano_mb:.2f} MB")
print("\nVista rápida de las primeras 3 filas:")
df_unido[["DIRECTORIO", "NOMBRE_DEPARTAMENTO", "ZONA", "PERCAPITA", "P1084", "FEX_C"]].head(3)

Archivo guardado exitosamente en: ../data/processed/dataset_brecha_unificado.csv
Dimensiones finales: 221,043 filas x 28 columnas
Tamaño del archivo unificado: 21.69 MB

Vista rápida de las primeras 3 filas:


,DIRECTORIO,NOMBRE_DEPARTAMENTO,ZONA,PERCAPITA,P1084,FEX_C
0,8369550,Bolívar,Urbano,747958.3333,1,472.131901
1,8369550,Bolívar,Urbano,747958.3333,1,472.131901
2,8369550,Bolívar,Urbano,747958.3333,1,472.131901
